# 07 - Regulatory Compliance Agent

An agentic system that **extracts regulatory requirements, audits company policies, and generates gap-analysis reports** using OpenAI function-calling.

| Component | Detail |
|-----------|--------|
| Regulation | EU AI Act (embedded excerpts) |
| Policy | Sample enterprise AI policy |
| LLM | OpenAI GPT-4o-mini via function calling |
| Pattern | Sequential extraction + gap analysis loop |

## Why an Agent?

Regulatory compliance is a multi-step reasoning task that requires:

1. **Parsing dense legal text** into discrete, actionable requirements.
2. **Cross-referencing** each requirement against existing internal policies.
3. **Scoring alignment** with nuance -- partial compliance is common.
4. **Prioritizing gaps** by risk level and suggesting concrete remediations.

A static script cannot adapt when a requirement is ambiguous or when a policy partially addresses multiple requirements. An agent can reason about context, ask follow-up questions via tools, and produce a structured compliance report that a human auditor can review.

In [ ]:
%pip install openai pandas --quiet

In [ ]:
import json
import textwrap
from datetime import datetime

import pandas as pd
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"

## 1 - Source Data: EU AI Act Excerpts & Sample Policy

In [ ]:
# Key excerpts from the EU AI Act (publicly available regulation text)
REGULATIONS = {
    "eu_ai_act": {
        "title": "EU Artificial Intelligence Act",
        "articles": {
            "Article 9 - Risk Management": (
                "A risk management system shall be established, implemented, documented and maintained "
                "in relation to high-risk AI systems. The risk management system shall be a continuous "
                "iterative process planned and run throughout the entire lifecycle of a high-risk AI system, "
                "requiring regular systematic review and updating. It shall identify and analyse the known "
                "and reasonably foreseeable risks that the high-risk AI system can pose to health, safety "
                "or fundamental rights. Estimation and evaluation of risks shall consider the intended purpose "
                "and reasonably foreseeable misuse. Testing shall be performed against preliminarily defined "
                "metrics and probabilistic thresholds."
            ),
            "Article 10 - Data Governance": (
                "High-risk AI systems which make use of techniques involving the training of AI models "
                "with data shall be developed on the basis of training, validation and testing data sets that "
                "meet quality criteria. Training, validation and testing data sets shall be subject to data "
                "governance and management practices appropriate for the intended purpose. Data sets shall be "
                "relevant, sufficiently representative, and to the best extent possible, free of errors and "
                "complete in view of the intended purpose. They shall have the appropriate statistical properties."
            ),
            "Article 11 - Technical Documentation": (
                "The technical documentation of a high-risk AI system shall be drawn up before that system "
                "is placed on the market or put into service and shall be kept up-to-date. It shall contain, "
                "at a minimum, a general description of the AI system, a detailed description of the elements "
                "of the AI system and of the process for its development, detailed information about the "
                "monitoring, functioning and control of the AI system, a description of the appropriateness "
                "of the performance metrics, a description of the risk management system, and a description "
                "of relevant changes made over the life cycle."
            ),
            "Article 13 - Transparency": (
                "High-risk AI systems shall be designed and developed in such a way as to ensure that their "
                "operation is sufficiently transparent to enable deployers to interpret a system's output and "
                "use it appropriately. An appropriate type and degree of transparency shall be ensured, with "
                "a view to achieving compliance. High-risk AI systems shall be accompanied by instructions "
                "for use in an appropriate digital format or otherwise that include concise, complete, correct "
                "and clear information that is relevant, accessible and comprehensible to deployers."
            ),
            "Article 14 - Human Oversight": (
                "High-risk AI systems shall be designed and developed in such a way, including with appropriate "
                "human-machine interface tools, as to enable them to be effectively overseen by natural persons "
                "during the period in which they are in use. Human oversight shall aim to minimise the risks "
                "to health, safety or fundamental rights. The measures shall enable the individuals to fully "
                "understand the capacities and limitations of the AI system, to correctly interpret its output, "
                "and to decide not to use the system, to override or reverse its output."
            ),
            "Article 15 - Accuracy, Robustness, Cybersecurity": (
                "High-risk AI systems shall be designed and developed in such a way that they achieve an "
                "appropriate level of accuracy, robustness and cybersecurity, and that they perform consistently "
                "in those respects throughout their lifecycle. The levels of accuracy and the relevant accuracy "
                "metrics shall be declared in the accompanying instructions of use. High-risk AI systems shall "
                "be resilient against attempts by unauthorised third parties to alter their use, outputs or "
                "performance by exploiting system vulnerabilities."
            ),
        }
    }
}

# Sample company AI policy (deliberately incomplete for the agent to find gaps)
COMPANY_POLICY = """
ACME Corp AI Governance Policy v2.3 (2025-06-01)

1. Purpose: This policy governs the development and deployment of AI systems at ACME Corp.

2. Risk Classification: All AI projects must be classified as low, medium, or high risk
   during the design phase. High-risk systems require additional review.

3. Data Requirements: Training data must be documented with source, date collected, and
   size. Data quality checks are recommended but not mandatory.

4. Model Documentation: A model card must be produced for all production AI systems,
   including architecture, training data summary, and performance metrics on test sets.

5. Fairness: Models must be tested for bias across protected categories (race, gender, age)
   before deployment. Results are logged but no minimum thresholds are defined.

6. Monitoring: Production systems are monitored for uptime and latency. Drift detection
   is planned for future implementation.

7. Human Review: High-risk decisions (credit, hiring) require a human in the loop.
   The definition of 'high-risk decision' is left to business unit leads.

8. Security: Standard IT security policies apply. No AI-specific cybersecurity measures.

9. Incident Response: AI incidents are handled through the general IT incident process.
   No AI-specific escalation path exists.
"""

print(f"Loaded {len(REGULATIONS['eu_ai_act']['articles'])} regulation articles")
print(f"Company policy: {len(COMPANY_POLICY)} characters")

## 2 - Define Tools

In [ ]:
state = {
    "requirements": [],
    "alignments": [],
    "gaps": [],
    "remediations": [],
}


def load_regulation(name: str) -> str:
    """Load a regulation by name and return its full text with article titles."""
    reg = REGULATIONS.get(name)
    if not reg:
        return json.dumps({"error": f"Unknown regulation: {name}", "available": list(REGULATIONS.keys())})
    result = {"title": reg["title"], "articles": {}}
    for art_name, art_text in reg["articles"].items():
        result["articles"][art_name] = art_text
    return json.dumps(result)


def extract_requirements(regulation_text: str, category: str) -> str:
    """Parse regulation text and extract discrete requirements for a category.
    The agent provides the text and category; this tool structures the output."""
    # In practice the LLM does the extraction; this tool records results.
    req_id = f"REQ-{len(state['requirements'])+1:03d}"
    req = {
        "id": req_id,
        "category": category,
        "source_text": regulation_text[:500],
        "status": "extracted",
    }
    state["requirements"].append(req)
    return json.dumps({"requirement_id": req_id, "category": category,
                       "instruction": "Now check this requirement against the company policy."})


def check_policy_alignment(requirement: str, policy_text: str) -> str:
    """Check how well a specific requirement is addressed by the company policy.
    Returns alignment analysis with a score."""
    # Simulate structured alignment check
    alignment = {
        "requirement_summary": requirement[:200],
        "policy_excerpt": policy_text[:300] if policy_text else "No matching policy section found",
        "instruction": (
            "Evaluate alignment on a scale of 0-100. "
            "0 = not addressed at all, 50 = partially addressed, 100 = fully addressed. "
            "Provide your score and reasoning, then call identify_gap if score < 80."
        )
    }
    state["alignments"].append(alignment)
    return json.dumps(alignment)


def identify_gap(requirement: str, alignment_score: int) -> str:
    """Record a compliance gap with severity based on alignment score."""
    if alignment_score >= 80:
        severity = "low"
    elif alignment_score >= 50:
        severity = "medium"
    elif alignment_score >= 20:
        severity = "high"
    else:
        severity = "critical"

    gap = {
        "gap_id": f"GAP-{len(state['gaps'])+1:03d}",
        "requirement": requirement[:200],
        "alignment_score": alignment_score,
        "severity": severity,
        "status": "identified",
    }
    state["gaps"].append(gap)
    return json.dumps(gap)


def suggest_remediation(gap: str) -> str:
    """Record a remediation suggestion for a gap. The LLM provides the actual suggestion."""
    remediation = {
        "remediation_id": f"REM-{len(state['remediations'])+1:03d}",
        "gap_reference": gap[:200],
        "status": "suggested",
        "instruction": (
            "Provide: 1) specific policy changes needed, 2) implementation priority (P1-P4), "
            "3) estimated effort (low/medium/high), 4) responsible team."
        )
    }
    state["remediations"].append(remediation)
    return json.dumps(remediation)

## 3 - Tool Dispatcher & Schemas

In [ ]:
TOOL_MAP = {
    "load_regulation": load_regulation,
    "extract_requirements": extract_requirements,
    "check_policy_alignment": check_policy_alignment,
    "identify_gap": identify_gap,
    "suggest_remediation": suggest_remediation,
}


def dispatch_tool(name: str, arguments: dict) -> str:
    fn = TOOL_MAP.get(name)
    if fn is None:
        return json.dumps({"error": f"Unknown tool: {name}"})
    try:
        return fn(**arguments)
    except Exception as e:
        return json.dumps({"error": str(e)})


tools = [
    {
        "type": "function",
        "function": {
            "name": "load_regulation",
            "description": "Load a regulation by name. Returns articles and full text.",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Regulation identifier (e.g., 'eu_ai_act')"}
                },
                "required": ["name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "extract_requirements",
            "description": "Extract and record a discrete compliance requirement from regulation text.",
            "parameters": {
                "type": "object",
                "properties": {
                    "regulation_text": {"type": "string", "description": "The regulation article text to extract from"},
                    "category": {"type": "string", "description": "Requirement category (e.g., 'risk_management', 'data_governance', 'transparency')"}
                },
                "required": ["regulation_text", "category"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_policy_alignment",
            "description": "Check how well a requirement is addressed by the company policy.",
            "parameters": {
                "type": "object",
                "properties": {
                    "requirement": {"type": "string", "description": "The compliance requirement to check"},
                    "policy_text": {"type": "string", "description": "Relevant section of company policy"}
                },
                "required": ["requirement", "policy_text"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "identify_gap",
            "description": "Record a compliance gap when alignment score is below threshold.",
            "parameters": {
                "type": "object",
                "properties": {
                    "requirement": {"type": "string", "description": "The requirement that has a gap"},
                    "alignment_score": {"type": "integer", "description": "Alignment score 0-100"}
                },
                "required": ["requirement", "alignment_score"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "suggest_remediation",
            "description": "Suggest remediation for a compliance gap.",
            "parameters": {
                "type": "object",
                "properties": {
                    "gap": {"type": "string", "description": "Description of the compliance gap to remediate"}
                },
                "required": ["gap"]
            }
        }
    }
]

## 4 - Agent Loop

In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""\
You are a Regulatory Compliance Agent. Your job is to audit a company's AI policy
against the EU AI Act and produce a gap analysis report.

Company policy:
{policy}

Workflow:
1. Load the EU AI Act regulation using load_regulation('eu_ai_act').
2. For each article, extract the key requirements using extract_requirements.
3. For each requirement, check alignment against the company policy using check_policy_alignment.
4. If alignment score < 80, call identify_gap to record the gap.
5. For each gap, call suggest_remediation.
6. After processing all articles, summarize findings in a compliance report.
   Include: total requirements checked, gaps found by severity, and top remediation priorities.
7. End your final message with DONE.

Be thorough but concise. Focus on actionable findings.
""")


def run_agent(max_turns: int = 30):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT.format(policy=COMPANY_POLICY)},
        {"role": "user", "content": (
            "Please audit our AI policy against the EU AI Act. "
            "Start by loading the regulation, then work through each article systematically."
        )}
    ]

    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            print(f"\n[Turn {turn+1}] Agent says:\n{msg.content[:500]}")
            if msg.content and "DONE" in msg.content.upper():
                break
            continue

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            print(f"[Turn {turn+1}] Tool: {tc.function.name}({list(args.keys())})")
            result = dispatch_tool(tc.function.name, args)
            print(f"         Result: {result[:150]}...")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

    return messages

In [ ]:
conversation = run_agent(max_turns=30)

## 5 - Results Analysis

In [ ]:
print(f"Requirements extracted: {len(state['requirements'])}")
print(f"Alignment checks: {len(state['alignments'])}")
print(f"Gaps identified: {len(state['gaps'])}")
print(f"Remediations suggested: {len(state['remediations'])}")

In [ ]:
if state["gaps"]:
    gaps_df = pd.DataFrame(state["gaps"])
    print("\nGaps by Severity:")
    print(gaps_df["severity"].value_counts().to_string())
    print("\nDetailed Gap List:")
    for _, row in gaps_df.iterrows():
        print(f"  [{row['severity'].upper()}] {row['gap_id']}: "
              f"{row['requirement'][:80]}... (score: {row['alignment_score']})")
else:
    print("No gaps were identified.")

In [ ]:
import matplotlib.pyplot as plt

if state["gaps"]:
    gaps_df = pd.DataFrame(state["gaps"])
    severity_order = ["critical", "high", "medium", "low"]
    colors = {"critical": "#d32f2f", "high": "#f57c00", "medium": "#fbc02d", "low": "#388e3c"}

    counts = gaps_df["severity"].value_counts()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Severity distribution
    bars = ax1.bar(
        [s for s in severity_order if s in counts.index],
        [counts.get(s, 0) for s in severity_order if s in counts.index],
        color=[colors[s] for s in severity_order if s in counts.index]
    )
    ax1.set_title("Gaps by Severity")
    ax1.set_ylabel("Count")

    # Alignment scores
    ax2.hist(gaps_df["alignment_score"], bins=10, range=(0, 100), edgecolor="black")
    ax2.axvline(x=80, color="red", linestyle="--", label="Compliance threshold")
    ax2.set_title("Alignment Score Distribution")
    ax2.set_xlabel("Score")
    ax2.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Extract the final compliance report from conversation
final_messages = [m for m in conversation if hasattr(m, 'content') and m.content and "DONE" in (m.content or "").upper()]
if not final_messages:
    # Check dict-style messages too
    final_messages = [m for m in conversation if isinstance(m, dict) and "DONE" in (m.get("content") or "").upper()]

if final_messages:
    report = final_messages[-1].content if hasattr(final_messages[-1], 'content') else final_messages[-1]["content"]
    print("=" * 60)
    print("COMPLIANCE REPORT")
    print("=" * 60)
    print(report)
else:
    print("Agent did not produce a final report. Review the conversation above.")

## 6 - Key Takeaways

1. **Structured extraction**: The agent decomposed dense legal text into discrete, auditable requirements.
2. **Gap prioritization**: Severity ratings enable the compliance team to focus on critical and high gaps first.
3. **Actionable remediations**: Each gap links to a concrete remediation with effort estimates.
4. **Reproducibility**: The tool call log provides a full audit trail of the analysis.
5. **Extensibility**: Adding new regulations (NIST AI RMF, ISO 42001) only requires adding entries to the `REGULATIONS` dict.